<a href="https://colab.research.google.com/github/sp6jaz/python2-materialy/blob/master/dzienne/tydzien-11/wyklad/stats_demo.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

**Dwie ścieżki pracy:**
- **Lokalna (zalecana)**: VS Code + venv + Jupyter — pełna kontrola, kod zostaje na Twoim komputerze.
- **Colab (kliknij i działa)**: środowisko Google gotowe w 5 sekund, działa nawet z telefonu. Kliknij ikonę powyżej.

# Statystyka opisowa w Pythonie

**Programowanie w Pythonie II** | Wykład 11
**Politechnika Opolska** | Analityka danych w biznesie

---

## Co dzisiaj?

Dotąd umieliście **wczytywać** dane (W05), **czyścić** (W07), **łączyć** (W08), **wizualizować** (W09–W10). Mieliście dane, mieliście wykres. Ładnie wygląda — **ale co to znaczy?**

Czy wynagrodzenia w firmie są wysokie? Czy są fair? Czy pracownicy z dłuższym stażem zarabiają proporcjonalnie więcej? Czy są osoby, których zarobki to anomalia (statystycznie odstające — *outlier*, dosłownie „leżący na zewnątrz”)?

To są pytania, na które odpowiada **statystyka opisowa** (deskryptywna — opisująca dane, nie wnioskująca o populacji). Nie ML (machine learning — uczenie maszynowe), nie deep learning (głębokie sieci neuronowe) — zwykłe liczby: średnia, mediana, odchylenie standardowe, korelacja. Te liczby mówią menedżerowi więcej niż sto wykresów.

```mermaid
graph TD
    A["Dane biznesowe"] --> B["Tendencja centralna<br/>średnia, mediana, dominanta"]
    A --> C["Rozproszenie<br/>std, IQR, percentyle"]
    A --> D["Zależności<br/>korelacja Pearson, Spearman"]
    A --> E["Kształt rozkładu<br/>skośność, kurtoza"]
    A --> F["Anomalie<br/>outliery: IQR, z-score"]
    B --> G["Decyzja biznesowa"]
    C --> G
    D --> G
    E --> G
    F --> G
```

### Po tym wykładzie potrafisz:

1. **Wyjaśnić** różnicę między miarami centralnymi (średnia, mediana, dominanta) i wskazać kiedy która jest właściwa
2. **Obliczać** miary rozproszenia (`std`, IQR, percentyle, rozstęp) i interpretować je biznesowo
3. **Stosować** korelację Pearsona (`np.corrcoef`) i Spearmana (`scipy.stats.spearmanr`) — z poprawnym wyborem dla danych liniowych vs monotonicznych (rosnących/malejących razem, niezależnie od kształtu krzywej)
4. **Wykrywać** wartości odstające (outliery) metodą IQR (rozstępu ćwiartkowego) i z-score (standaryzacji), oceniać ich wpływ na średnią/std
5. **Charakteryzować** rozkład danych (skośność, kurtoza) i decydować jakie miary są właściwe
6. **Oceniać krytycznie** twierdzenia statystyczne (korelacja ≠ przyczynowość)

**Dokumentacja:** [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html) | [numpy statistics](https://numpy.org/doc/stable/reference/routines.statistics.html) | [pandas describe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)

---

## 0. Po co statystyka opisowa? Datasaurus Dozen

Zanim policzymy cokolwiek — pokażę dlaczego **same liczby kłamią**. Pamiętacie z W09 *Anscombe quartet* — cztery datasety o identycznych statystykach, ale kompletnie różnych wykresach? Statystyka opisowa ma **swoją własną** wersję tego paradoksu.

**Datasaurus Dozen** (Matejka & Fitzmaurice 2017) — 13 datasetów (zbiorów danych: dinozaur, gwiazdka, ślimak...) — **wszystkie mają identyczną średnią, identyczne odchylenie standardowe i identyczną korelację**. Wyglądają tak różnie jak tylko możliwe.

**Wniosek:** statystyka opisowa **NIE zastępuje wykresu**. Ona go **uzupełnia**. Trzy zasady na dziś:

1. **Każda liczba ma kontekst** — średnia 8500 zł brzmi inaczej dla pracownika sprzątającego niż dla CEO.
2. **Sama liczba nie wystarczy** — średnia bez rozproszenia, korelacja bez scatter plotu = niepełne informacje.
3. **Statystyka domyka decyzję** — nie pytasz „ile to jest”, tylko „co z tym zrobić” (zatrudnić? podnieść? zbadać outlier?).

> *„The first principle is that you must not fool yourself — and you are the easiest person to fool.”*
> *(„Pierwszą zasadą jest nie oszukiwać samego siebie — a sam jesteś osobą, którą najłatwiej oszukać.”)*
> — Richard Feynman, *Cargo Cult Science* (Caltech 1974)

> **Ciekawostka 1:** 100 lat temu (1921) Frank Anscombe miał 3 lata. Pokazał swój *quartet* w 1973. Datasaurus to *Anscombe XXI wieku* — pokazuje że nawet rozszerzenie zestawu liczb (do 5: średnia + std + r) nie wystarczy.

> **Ciekawostka 2:** Reguła „outlier = poza [Q1 − 1.5·IQR, Q3 + 1.5·IQR]” pochodzi od Johna Tukeya (książka *Exploratory Data Analysis*, 1977) — tego samego, którego cytujemy na końcu wykładu. Wartość 1.5 nie jest „magiczna” — Tukey zaproponował ją jako wygodny próg pomiędzy 1 a 2. Dla rozkładu normalnego wypada wówczas poza wąsami ~0.7% obserwacji (~7 na 1000); dla danych skośnych odsetek bywa wyższy.

In [ ]:
# Datasaurus Dozen — pokaż że identyczne statystyki nie znaczą identyczne dane
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Generujemy 4 datasety o identycznych statystykach (uproszczony Datasaurus)
np.random.seed(42)
n = 142

# Dataset 1: chmura (bivariate normal)
x1 = np.random.normal(54, 16, n)
y1 = np.random.normal(48, 26, n)

# Dataset 2: linia (silnie liniowy)
x2 = np.random.uniform(20, 90, n)
y2 = 0.5 * x2 + np.random.normal(20, 5, n)
# Skaluj zeby dac identyczne mean/std jako dataset 1
y2 = (y2 - y2.mean()) / y2.std() * 26 + 48
x2 = (x2 - x2.mean()) / x2.std() * 16 + 54

# Dataset 3: dwie chmury (bimodalny)
x3a = np.random.normal(35, 5, n // 2)
x3b = np.random.normal(73, 5, n - n // 2)
y3a = np.random.normal(48, 26, n // 2)
y3b = np.random.normal(48, 26, n - n // 2)
x3 = np.concatenate([x3a, x3b])
y3 = np.concatenate([y3a, y3b])
x3 = (x3 - x3.mean()) / x3.std() * 16 + 54
y3 = (y3 - y3.mean()) / y3.std() * 26 + 48

# Dataset 4: 'V' (nieliniowy)
xv = np.linspace(20, 90, n)
yv = np.abs(xv - 55) * 1.5 + np.random.normal(0, 5, n)
yv = (yv - yv.mean()) / yv.std() * 26 + 48
xv = (xv - xv.mean()) / xv.std() * 16 + 54

datasets = [("Chmura", x1, y1), ("Linia", x2, y2),
            ("Dwie chmury", x3, y3), ("Litera V", xv, yv)]

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (name, x, y) in zip(axes.flat, datasets):
    ax.scatter(x, y, alpha=0.6, s=40, edgecolor='white', linewidth=0.5)
    r, _ = stats.pearsonr(x, y)
    ax.set_title(f"{name}\nśrednia x={x.mean():.1f} y={y.mean():.1f} | std x={x.std():.1f} y={y.std():.1f} | r={r:+.2f}",
                 fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(alpha=0.3)
fig.suptitle('Datasaurus-style: niemal identyczne statystyki, totalnie różne dane', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Co tu się stało?** Cztery wykresy mają **niemal identyczne** średnie, std i korelacje (drobne różnice rzędu 0.1 wynikają z naszego uproszczenia — pełen Datasaurus Dozen Matejki i Fitzmaurice'a 2017 ma identyczne **co do drugiej cyfry po przecinku**). Wyglądają zupełnie różnie. Gdybyście dostali tylko statystyki — uznalibyście je za „te same dane”. To by była katastrofa biznesowa.

**Złota zasada** (powraca z W09): **statystyka opisowa + wizualizacja idą razem.** Liczby bez wykresu kłamią, wykres bez liczb jest niepełny.

Dziś budujemy ten zestaw: **liczby, które rozumiesz**.

---

## 1. Setup i dataset HR — 200 pracowników

Cały wykład pracujemy na danych HR (działu kadr) firmy: **200 pracowników**, **5 działów** (IT, Sprzedaż, HR, Marketing, Finanse), z atrybutami: staż pracy (lata), wynagrodzenie (PLN), ocena roczna (1–5).

Dataset jest **syntetyczny** (generowany w notebooku), ale wzorowany na realnych strukturach firm — w tym celowo zawiera:
- prawostronną skośność wynagrodzeń (długi ogon prezesów),
- pozytywną korelację staż ↔ wynagrodzenie (ale nie liniową — z saturacją),
- 2-3 outliery (anomalie: zarobki znacznie poza rozkładem).

In [ ]:
# Wersje pakietów
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import scipy

print(f"numpy:  {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"scipy:  {scipy.__version__}")

In [ ]:
# Generowanie realistycznego datasetu HR (kadrowego)
np.random.seed(42)  # reprodukowalność
n = 200

# Działy z prawdopodobieństwami (IT najwięcej)
dzialy = np.random.choice(
    ['IT', 'Sprzedaz', 'HR', 'Marketing', 'Finanse'], n,
    p=[0.30, 0.25, 0.15, 0.20, 0.10]
)

# Staż pracy (lata) z rozkładu gamma — realistyczny, prawostronnie skośny
staz = np.random.gamma(shape=3, scale=2, size=n).clip(0.5, 20).round(1)

# Wynagrodzenie: baza wg działu + premia za staż + szum
baza = {'IT': 9000, 'Sprzedaz': 7000, 'HR': 6500, 'Marketing': 7500, 'Finanse': 8500}
wynagrodzenie = np.array([
    baza[d] + staz[i] * 300 + np.random.normal(0, 1200)
    for i, d in enumerate(dzialy)
]).clip(4000, 60000).round(0)

# 3 outliery — rzeczywiste anomalie (kierownictwo)
wynagrodzenie[5] = 32000   # CEO (Chief Executive Officer — prezes zarządu)
wynagrodzenie[42] = 28000  # CTO (Chief Technology Officer — dyrektor ds. technologii)
wynagrodzenie[101] = 25000 # CFO (Chief Financial Officer — dyrektor finansowy)

# Ocena roczna 1-5 (rozklad ~normalny obciety)
ocena = np.random.normal(3.8, 0.7, n).clip(1, 5).round(1)

df = pd.DataFrame({
    'dzial': dzialy,
    'staz_lat': staz,
    'wynagrodzenie': wynagrodzenie.astype(int),
    'ocena': ocena,
})

print(f"Shape: {df.shape}")
df.head(8)

**🔑 Kluczowe rozróżnienie: parametr populacji vs estymator z próby**

Te 200 pracowników to **próba** (*sample* — podzbiór, który faktycznie zmierzyliśmy). Gdyby firma miała 10 000 osób, to byłaby **populacja** (wszystkie obiekty, o których chcemy wnioskować). Statystyki które za chwilę policzymy (`x̄`, `s`) to **estymatory** (przybliżenia — liczone z próby, nieznanych dokładnie) — przybliżenia prawdziwych parametrów populacji (`μ`, `σ`).

**Analogia Excel:** Excel `=AVERAGE(...)` — średnia próby. Nigdy nie znamy μ całej populacji ludzi w branży, znamy tylko `x̄` z konkretnego pliku.

**Ważny szczegół Pythona — stopnie swobody (degrees of freedom — `ddof`):**
- `np.std(arr)` → `ddof=0` (domyślnie) → std **populacji** (dziel przez n)
- `np.std(arr, ddof=1)` → std **próby** (dziel przez n–1, *Bessel correction* — poprawka Bessela)
- `pandas.Series.std()` → domyślnie `ddof=1` → próba (dlatego pandas jest bezpieczniejszy dla danych biznesowych)

**Intuicja `n–1` (poprawka Bessela):** średnia próby `x̄` jest sama estymatorem — liczymy ją z tych samych danych co std. „Tracimy” więc 1 stopień swobody. Dzielenie przez `n–1` zamiast `n` eliminuje systematyczne zaniżenie wariancji próby.

Dla n=200 różnica jest mała (~0.25%), dla n=10 — duża (~5%).

In [ ]:
# Demonstracja: ddof=0 vs ddof=1 dla wynagrodzenia
placa = df['wynagrodzenie']

std_populacji  = np.std(placa, ddof=0)
std_proby      = np.std(placa, ddof=1)
std_pandas     = placa.std()  # domyślnie ddof=1

print(f"np.std(ddof=0) — populacja:    {std_populacji:>10,.2f}")
print(f"np.std(ddof=1) — próba:        {std_proby:>10,.2f}")
print(f"pandas .std() — domyślnie n-1: {std_pandas:>10,.2f}")
print(f"\nRóżnica próba vs populacja:    {(std_proby - std_populacji):>10,.2f} ({(std_proby/std_populacji - 1)*100:.2f}%)")

---

## 2. Tendencja centralna — średnia, mediana, dominanta

**Pytanie biznesowe:** *Ile zarabia „typowy” pracownik?*

Odpowiedź zależy od tego, **co rozumiemy przez „typowy”**. Trzy miary dają trzy różne odpowiedzi:

| Miara | Definicja | Kiedy użyć |
|-------|-----------|------------|
| **Średnia** (`mean`) | Suma ÷ liczba | Dane symetryczne, bez outlierów |
| **Mediana** (`median`) | Środkowa wartość po posortowaniu | Dane skośne lub z outlierami |
| **Dominanta** (`mode`) | Wartość najczęstsza | Dane kategoryczne lub dyskretne |

**🚨 Częsty błąd interpretacyjny:** „średnia = mediana = wartość typowa” — to NIEPRAWDA dla danych biznesowych. Wynagrodzenia, ceny domów, czas oczekiwania na infolinii — niemal zawsze skośne. Wtedy średnia ≠ mediana, i mediana jest właściwa.

In [ ]:
# Trzy miary centralne dla wynagrodzeń
placa = df['wynagrodzenie']

srednia   = placa.mean()
mediana   = placa.median()
# Dominanta — wartość najczęstsza. Dla danych ciągłych nie ma sensu (każda jest unikalna),
# więc grupujemy do tysięcy zł
dominanta = (placa // 1000 * 1000).mode().iloc[0]

print("=" * 50)
print("   MIARY TENDENCJI CENTRALNEJ — wynagrodzenia")
print("=" * 50)
print(f"  Średnia:    {srednia:>10,.0f} PLN")
print(f"  Mediana:    {mediana:>10,.0f} PLN")
print(f"  Dominanta:  {dominanta:>10,.0f} PLN  (zaokr. do 1k)")
print("=" * 50)

# Diagnoza skośności na podstawie różnicy mean - median
if mediana < srednia:
    roznica = srednia - mediana
    print(f"\nMediana < Średnia (różnica: {roznica:,.0f} PLN)")
    print("→ Rozkład prawdopodobnie PRAWOSTRONNIE SKOŚNY (długi ogon w prawo)")
    print("   (heurystyka działa dla rozkładów unimodalnych — z jednym pikiem)")
    print("→ Outliery wysokich pensji podnoszą średnią")
    print("→ MEDIANA jest bezpieczniejsza do raportu")
else:
    print("\nMediana ~ Średnia → rozkład symetryczny")

**Co widać:** średnia (8800 zł) jest wyższa od mediany (7500 zł) o około 1300 zł. To **prawostronna skośność** — ogon wysokich pensji (CEO, CTO, CFO) podnosi średnią, ale **nie wpływa na medianę**.

**Skutek biznesowy:**
- W raporcie dla zarządu lub mediów: **podaj medianę** („typowa pensja w firmie X to 7500 zł”)
- W obliczeniach budżetowych (płacenie podatków, ZUS): **podaj średnią** (bo suma się liczy)
- W rozmowach z kandydatami: **podaj zakres** (P25–P75, czyli pas IQR — sekcja 3)

In [ ]:
# Wizualizacja: rozkład + 3 miary centralne
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(placa, bins=30, color='#5B9BD5', alpha=0.7, edgecolor='white')
ax.axvline(srednia, color='red', linestyle='--', linewidth=2, label=f'Średnia: {srednia:,.0f}')
ax.axvline(mediana, color='green', linestyle='-', linewidth=2, label=f'Mediana: {mediana:,.0f}')
ax.axvline(dominanta, color='purple', linestyle=':', linewidth=2, label=f'Dominanta: {dominanta:,.0f}')
ax.set_xlabel('Wynagrodzenie (PLN)')
ax.set_ylabel('Liczba pracowników')
ax.set_title('Rozkład wynagrodzeń — gdzie leży typowa pensja?')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Na wykresie widać wyraźnie **długi ogon w prawo** — to outliery (3 osoby: CEO/CTO/CFO). Mediana jest „w środku gęstości”, średnia się przesuwa za ogonem.

### Mini-ćwiczenie A — branża e-commerce (przykład gotowy)

Wyobraź sobie sklep internetowy. Średnia wartość koszyka = 320 zł, mediana = 180 zł. Co to mówi o klientach? Jak zaprojektujesz ofertę specjalną?

In [ ]:
# Przykład: rozkład wartości koszyka w e-commerce (skośny!)
np.random.seed(0)
koszyki = np.concatenate([
    np.random.gamma(shape=2, scale=80, size=950),  # standardowi klienci
    np.random.uniform(800, 3500, 50),               # 50 'whale' clients (top 5%)
])

print(f"  Średnia koszyka: {koszyki.mean():>7,.0f} zł")
print(f"  Mediana koszyka: {np.median(koszyki):>7,.0f} zł")
print()
print("Wniosek biznesowy:")
print("- Średnia (~320) sugeruje 'sklep premium' — fałszywie")
print("- Mediana (~180) — typowy klient kupuje za 180 zł")
print("- 50 'whales' (5%) podnosi średnią; oferta dla 95% powinna celować w 180 zł")

### Mini-ćwiczenie B — rozkład czasu odpowiedzi infolinii

Uzupełnij brakujące fragmenty. Mamy logi infolinii — czas oczekiwania w sekundach. Większość krótka, kilka długich (skośność).

In [ ]:
# Uzupełnij brakujące fragmenty oznaczone ___
np.random.seed(7)
czasy_oczek = np.concatenate([
    np.random.exponential(scale=30, size=480),   # ___ (większość: szybka odpowiedź)
    np.random.uniform(180, 600, 20),              # ___ (długie czekanie — problemy)
])

# Policz średnią i medianę
sr_czas  = czasy_oczek.mean()
med_czas = np.median(czasy_oczek)

# ___ — wybierz miarę do raportu dla menedżera
print(f"Średnia: {sr_czas:.1f} s")
print(f"Mediana: {med_czas:.1f} s")
print()
# Analogiczna decyzja jak dla pensji — jakie zalecenie podajesz?

### Mini-ćwiczenie C — marketing: kliknięcia w reklamę

Dane: liczba kliknięć w bannery reklamowe na 1000 użytkowników. **Bez podpowiedzi** — wybierz miarę i uzasadnij.

In [ ]:
# Sam wybierz miarę i uzasadnij wybór
np.random.seed(11)
klikniecia = np.concatenate([
    np.zeros(700, dtype=int),               # 700 nieaktywnych
    np.random.poisson(2, 250),              # 250 normalnych
    np.random.poisson(15, 50),              # 50 zaangażowanych
])

# TODO: oblicz średnią, medianę, dominantę
# TODO: zinterpretuj. Co raportujesz w prezentacji wyników kampanii?
# Wskazówka: dominanta ma sens dla danych dyskretnych (przyjmujących policzalne wartości — np. 0, 1, 2, 3 kliknięć)

---

## 3. Rozproszenie — std, IQR, percentyle

**Pytanie biznesowe:** *Czy wynagrodzenia w firmie są stabilne, czy bardzo zróżnicowane?*

Sama tendencja centralna nie wystarczy. Trzy działy mogą mieć identyczną średnią pensję, ale jeden płaci wszystkim podobnie, drugi ma duże różnice. **Rozproszenie** to drugi wymiar opisu.

**🔑 Kluczowe rozróżnienie: centrum + rozproszenie idą razem.** Sama średnia bez std = oszustwo. „Średnia ocena = 4.2” może znaczyć:
- (4.2 ± 0.3): wszyscy zgadzają się że dobra → produkt **stabilnie dobry**
- (4.2 ± 1.5): część kocha (5), część nienawidzi (2) → produkt **polaryzujący**

Dwa różne biznesy z identyczną średnią!

### Trzy podstawowe miary

| Miara | Wzór / metoda | Interpretacja |
|-------|---------------|---------------|
| **Wariancja** (`var`) | średnia kwadratów odchyleń od średniej | „Rozrzut” obserwacji wokół centrum (jednostka kwadratowa — trudna do interpretacji) |
| **Odchylenie standardowe** (`std`) | √wariancja | Typowa odległość od średniej |
| **Rozstęp ćwiartkowy IQR** | Q3 − Q1 | Zakres środkowych 50% danych — odporne na outliery |

In [ ]:
# Trzy miary rozproszenia dla pensji
placa = df['wynagrodzenie']

wariancja = placa.var()           # ddof=1 (próba) — pandas default
std       = placa.std()           # ddof=1
rozstep   = placa.max() - placa.min()
Q1 = placa.quantile(0.25)
Q3 = placa.quantile(0.75)
iqr = Q3 - Q1

print("=" * 55)
print("   ROZPROSZENIE — wynagrodzenia")
print("=" * 55)
print(f"  Wariancja:               {wariancja:>14,.0f} PLN²")
print(f"  Odch. standardowe (std): {std:>14,.0f} PLN")
print(f"  Rozstęp (max-min):       {rozstep:>14,.0f} PLN")
print(f"  IQR (Q3-Q1):             {iqr:>14,.0f} PLN")
print(f"  Q1 (25 percentyl):       {Q1:>14,.0f} PLN")
print(f"  Q3 (75 percentyl):       {Q3:>14,.0f} PLN")
print("=" * 55)

**🚨 Częsty błąd interpretacyjny:** „std opisuje pełen zakres wartości”.

Nie. Std to **typowa** odległość od średniej. Reguła **68-95-99.7** (tylko dla rozkładu normalnego!) mówi:
- ~68% danych mieści się w `mean ± 1·std`
- ~95% w `mean ± 2·std`
- ~99.7% w `mean ± 3·std`

Ale **dla rozkładów skośnych** (jak nasze pensje) ta reguła **nie działa**. Dlatego dla skośnych danych raportuj IQR, nie std.

**🚨 Częsty błąd interpretacyjny:** „percentyl 90 = 90% populacji to ja”.

**Poprawna definicja:** percentyl 90 (P90) = wartość, **poniżej której** znajduje się 90% danych. Jeśli P90 wynagrodzenia = 12 000 zł, to znaczy że 90% pracowników zarabia ≤ 12 000.

In [ ]:
# Wizualizacja IQR i reguły 68-95-99.7
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Lewy panel: rozkład pensji + IQR
ax = axes[0]
ax.hist(placa, bins=30, color='#5B9BD5', alpha=0.7, edgecolor='white')
ax.axvspan(Q1, Q3, alpha=0.25, color='green', label=f'IQR: {Q1:,.0f}–{Q3:,.0f}')
ax.axvline(placa.median(), color='green', linestyle='-', lw=2, label='Mediana')
ax.set_xlabel('Wynagrodzenie (PLN)')
ax.set_title('IQR — środkowe 50% danych')
ax.legend()
ax.grid(alpha=0.3)

# Prawy panel: rozkład normalny vs reguła 68-95-99.7
ax = axes[1]
x = np.linspace(-4, 4, 200)
y = stats.norm.pdf(x)
ax.plot(x, y, color='#222', lw=2)
ax.fill_between(x, 0, y, where=(np.abs(x) <= 1), alpha=0.4, color='green', label='±1σ ≈ 68%')
ax.fill_between(x, 0, y, where=(np.abs(x) <= 2) & (np.abs(x) > 1), alpha=0.3, color='orange', label='±2σ ≈ 95%')
ax.fill_between(x, 0, y, where=(np.abs(x) <= 3) & (np.abs(x) > 2), alpha=0.2, color='red', label='±3σ ≈ 99.7%')
ax.set_xlabel('Odchylenie od średniej (w jednostkach σ)')
ax.set_title('Reguła 68-95-99.7 — dla rozkładu normalnego')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Percentyle — pełne spektrum

In [ ]:
# Percentyle 5, 25, 50, 75, 95
percentyle = [5, 25, 50, 75, 95]
wartosci = [placa.quantile(p / 100) for p in percentyle]

print("PERCENTYLE WYNAGRODZEŃ:")
for p, v in zip(percentyle, wartosci):
    print(f"  P{p:>2}: {v:>10,.0f} PLN")
print()
print(f"Interpretacja:")
print(f"  P5  = {wartosci[0]:,.0f} → 5% zarabia mniej (najniżej opłacani)")
print(f"  P50 = {wartosci[2]:,.0f} → mediana (średni pracownik)")
print(f"  P95 = {wartosci[4]:,.0f} → 5% zarabia więcej (*top earners* — najwyżej zarabiający)")

### Mini-ćwiczenie — porównaj rozproszenie 5 działów

Który dział ma najbardziej **stabilne** wynagrodzenia (mała std), który najbardziej **zróżnicowane** (duża std)?

**Powtórka z W08** — `groupby` + `agg` (split-apply-combine) z named aggregation. Ten sam wzorzec, nowy temat — agregacje statystyczne zamiast biznesowych KPI.

In [ ]:
# Group by + std + median per dział
raport = df.groupby('dzial', observed=True).agg(
    liczba=('wynagrodzenie', 'count'),
    mediana_pln=('wynagrodzenie', 'median'),
    std_pln=('wynagrodzenie', 'std'),
    iqr_pln=('wynagrodzenie', lambda x: x.quantile(0.75) - x.quantile(0.25)),
).round(0).sort_values('std_pln', ascending=False)

print(raport)

### Mini-ćwiczenie A — branża logistyka (przykład gotowy)

Czas dostawy paczek (w godzinach) z dwóch magazynów. **Który magazyn jest bardziej przewidywalny?** — analiza std vs IQR.

In [ ]:
# Worked: czasy dostawy z dwóch magazynów
np.random.seed(3)
magazyn_a = np.random.normal(48, 5, 200).clip(20, 100)         # wąsko skupione
magazyn_b = np.concatenate([np.random.normal(48, 4, 180),
                            np.random.uniform(80, 120, 20)])    # zwykle szybkie + 20 awarii

for nazwa, dane in [('Magazyn A (regularny)', magazyn_a), ('Magazyn B (z awariami)', magazyn_b)]:
    s = pd.Series(dane)
    iqr = s.quantile(0.75) - s.quantile(0.25)
    print(f"{nazwa:35s}  Mediana={s.median():5.1f}h  Std={s.std():5.1f}h  IQR={iqr:4.1f}h")

print()
print('Wniosek: A i B mają zbliżoną medianę, ale Std B jest 2x większy — przez 20 awarii.')
print('Dla studenta: podaj IQR (bo odporny na te awarie). Dla operatora: podaj std (bo widzi ryzyko).')

### Mini-ćwiczenie B — branża HR (uzupełnij __)

Wiek pracowników w 3 zespołach. Uzupełnij obliczenia percentyli i porównaj **rozproszenie** zespołów.

In [ ]:
# Spróbuj sam zanim spojrzysz: uzupełnij percentyle
np.random.seed(5)
zespol_juniorow = np.random.normal(28, 3, 30).clip(22, 40).round()
zespol_mieszany = np.random.normal(38, 8, 30).clip(22, 60).round()
zespol_seniorow = np.random.normal(48, 5, 30).clip(35, 65).round()

for nazwa, dane in [('Juniorzy', zespol_juniorow),
                    ('Mieszany', zespol_mieszany),
                    ('Seniorzy', zespol_seniorow)]:
    s = pd.Series(dane)
    p10 = s.quantile(0.10)       # 10. percentyl — UZUPEŁNIJ TO
    p90 = s.quantile(0.90)       # 90. percentyl — UZUPEŁNIJ TO
    rozstep_80 = p90 - p10       # zakres środkowych 80%
    print(f"{nazwa:10s}  P10={p10:.0f}  P90={p90:.0f}  zakres 80% = {rozstep_80:.0f} lat")

# Pytanie: który zespół ma najszerszy zakres wieku 80%? Co to mówi o jego strukturze?

### Mini-ćwiczenie C — branża marketing (samodzielnie)

Czas spędzony na stronie internetowej (w sekundach) — masz 1000 sesji. Część użytkowników wpada na 5–30 s („bounce” — odbijają się szybko), część czyta artykuł 2–5 minut. **Wybierz miarę rozproszenia i uzasadnij.**

In [ ]:
# Sam wybierz miarę — std czy IQR? Uzasadnij
np.random.seed(11)
czasy_sesji = np.concatenate([
    np.random.uniform(5, 30, 600),       # 60% bounce
    np.random.normal(180, 40, 400),       # 40% rzeczywiste czytanie
]).clip(1, 600)

# TODO 1: oblicz średnią i medianę
# TODO 2: oblicz std i IQR
# TODO 3: w komentarzu odpowiedz: która miara rozproszenia jest właściwa i dlaczego?
# Wskazówka: rozkład jest dwumodalny — zastanów się jak to wpływa na std i IQR

---

## 4. Korelacja — Pearson vs Spearman

**Pytanie biznesowe:** *Czy pracownicy z dłuższym stażem zarabiają więcej?*

**Korelacja** to liczba opisująca, na ile dwie zmienne „idą razem” — mierzy **siłę** i **kierunek** związku. Wartość: **od –1 do +1**.

| Wartość r | Interpretacja |
|-----------|---------------|
| ±1.0 | Idealnie liniowa zależność |
| ±0.7 do ±0.9 | Silna |
| ±0.4 do ±0.6 | Umiarkowana |
| ±0.1 do ±0.3 | Słaba |
| 0 | Brak zależności **liniowej** (uwaga: patrz częsty błąd dalej) |

*Progi orientacyjne; w psychologii / naukach społecznych Cohen (1988) przyjmuje inne wartości: 0.1 / 0.3 / 0.5 jako słabą / umiarkowaną / silną.*

**🔑 Kluczowe rozróżnienie: Pearson vs Spearman**

**Pearson** mierzy zależność **liniową** (czy idą po prostej?). **Spearman** — **monotoniczną** (czy rosną/maleją razem, niezależnie od kształtu?).

| Sytuacja | Pearson | Spearman |
|----------|---------|----------|
| y = 2x + 5 (liniowo) | +1.0 | +1.0 |
| y = exp(x) (rośnie nieliniowo) | ~0.7 | +1.0 |
| y = x² dla x>0 | ~0.95 | +1.0 |
| y = sin(x) | bliskie 0 | bliskie 0 |
| Dane z rangami (pozycje w uporządkowanej skali, np. junior/mid/senior) | wątpliwe | właściwe |

In [ ]:
# Korelacja staż ↔ wynagrodzenie
from scipy.stats import pearsonr, spearmanr

r_p, p_p = pearsonr(df['staz_lat'], df['wynagrodzenie'])
r_s, p_s = spearmanr(df['staz_lat'], df['wynagrodzenie'])

print(f"Korelacja staż ↔ wynagrodzenie:")
print(f"  Pearson  r = {r_p:+.3f}  (p-value = {p_p:.2e})  # p-value: prawdopodobieństwo otrzymania równie skrajnego r przypadkiem")
print(f"  Spearman r = {r_s:+.3f}  (p-value = {p_s:.2e})")
print()
if r_s > r_p + 0.05:
    print("→ Spearman > Pearson o ponad 0.05 → związek MONOTONICZNY ale nieliniowy")
    print("   (np. saturacja: po 10 latach staż przestaje podnosić pensję — *saturacja* czyli nasycenie)")
elif abs(r_p - r_s) < 0.05:
    print("→ Pearson ≈ Spearman → związek liniowy")

**🚨 Częsty błąd interpretacyjny:** „Pearson zawsze wykryje korelację, jeśli ona istnieje”.

Nie! Pearson wykryje tylko zależność **liniową**. Klasyczny kontrprzykład:

In [ ]:
# Pearson nie wykrywa zależności nieliniowych
# Zakres -3..3 dla liniowej i sinusa; 0..3 dla paraboli (zeby byla monotoniczna)
x_sym = np.linspace(-3, 3, 100)        # zakres symetryczny
x_pos = np.linspace(0.1, 3, 100)       # zakres tylko dodatni
np.random.seed(0)
y_lin   = 2 * x_sym + np.random.normal(0, 0.3, 100)
y_kwad_sym = x_sym**2 + np.random.normal(0, 0.3, 100)  # NIEMONOTONICZNA na [-3,3]
y_kwad_pos = x_pos**2 + np.random.normal(0, 0.3, 100)  # monotoniczna na [0,3]
y_sin   = np.sin(x_sym * 2) + np.random.normal(0, 0.2, 100)

tests = [
    ("liniowa y=2x na [-3,3]",           x_sym, y_lin),
    ("parabola y=x² na [-3,3] (niemonot.)", x_sym, y_kwad_sym),
    ("parabola y=x² na [0,3] (monoton.)",  x_pos, y_kwad_pos),
    ("sin y=sin(2x)",                     x_sym, y_sin),
]
for nazwa, xi, yi in tests:
    rp_test, _ = pearsonr(xi, yi)
    rs_test, _ = spearmanr(xi, yi)
    print(f"{nazwa:>40s}  Pearson={rp_test:+.2f}  Spearman={rs_test:+.2f}")

**Co widać**:
- Liniowa: oba ~+1 (idealna prosta)
- Parabola na [-3,3] (niemonotoniczna): oba ~0 — funkcja maleje na [-3,0], rośnie na [0,3], więc *ani* Pearson *ani* Spearman nie wykrywają zależności
- Parabola na [0,3] (monotoniczna): Pearson ~0.97 (lekko zakrzywione), Spearman ~1.00 (idealnie monotoniczne) — TO jest klasyczny przypadek przewagi Spearmana
- Sin: oba ~0 — prawdziwy brak zależności

**Złota zasada (Tukey):** zacznij od **scatter plotu**. Wybór Pearson/Spearman robi się patrząc na kształt — nie automatycznie.

**🚨 Częsty błąd interpretacyjny — KORELACJA ≠ PRZYCZYNOWOŚĆ**

To jest najczęstszy błąd interpretacyjny w mediach i raportach biznesowych. Klasyczne przykłady:

1. **Lody i utonięcia**: r ≈ 0.9 — bo lato (sezonowość). Lody nie powodują utonięć.
2. **Kawa i długie życie**: korelacja r ≈ 0.3. Kawa? Czy ludzie którzy stać na kawę = lepsza opieka medyczna?
3. **Reklama → sprzedaż**: r ≈ 0.7. Czy reklama POWODUJE? Czy może obie zmienne rosną razem (sezonowość)?

**Jak weryfikować przyczynowość:** randomizowane eksperymenty, A/B testy (eksperyment porównujący dwa warianty na losowo przydzielonych grupach ), kontrola zmiennych ukrytych (*confounders* — zmiennych zakłócających, np. pora roku jednocześnie wpływająca na lody i utonięcia).

> **Ciekawostka:** korelacja Pearsona została wprowadzona przez Karla Pearsona w 1896 roku w pracy *Mathematical Contributions to the Theory of Evolution* — ponad 130 lat temu. Pearson budował aparat statystyczny do biologii ewolucyjnej, ale dziś jego współczynnik to standard wszędzie: ekonomii, marketingu, medycynie, ML.

> **Klasyk metodologii:** „lody i utonięcia” to przykład spopularyzowany przez Tylera Vigena (*Spurious Correlations*, 2015 — strona [tylervigen.com/spurious-correlations](https://tylervigen.com/spurious-correlations) ma dziesiątki absurdalnych korelacji typu „liczba doktoratów z socjologii vs liczba zgonów od porażenia prądem”).

In [ ]:
# Scatter staż vs pensja + linia regresji
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['staz_lat'], df['wynagrodzenie'], alpha=0.6, s=40, edgecolor='white', linewidth=0.5)

# linia trendu — regresja liniowa (dopasowanie prostej minimalizującej sumę kwadratów odchyleń)
z = np.polyfit(df['staz_lat'], df['wynagrodzenie'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['staz_lat'].min(), df['staz_lat'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', lw=2, label=f'y = {z[0]:.0f}·x + {z[1]:.0f}')

ax.set_xlabel('Staż pracy (lata)')
ax.set_ylabel('Wynagrodzenie (PLN)')
ax.set_title(f'Staż vs wynagrodzenie  (Pearson r = {r_p:+.2f})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Mini-ćwiczenie A — branża sprzedaż (przykład gotowy)

Cena produktu vs liczba sprzedanych sztuk (krzywa popytu). Klasyczna **ujemna korelacja**: drożej → mniej sprzedaży.

In [ ]:
# Worked: krzywa popytu (cena vs ilość sprzedaży)
np.random.seed(8)
ceny = np.linspace(50, 500, 50)
ilosci = 1000 - 1.5 * ceny + np.random.normal(0, 40, 50)
ilosci = ilosci.clip(0, None)

rp_demo, _ = pearsonr(ceny, ilosci)
rs_demo, _ = spearmanr(ceny, ilosci)
print(f"Cena vs sprzedaż:  Pearson={rp_demo:+.3f}  Spearman={rs_demo:+.3f}")
print()
print('Wniosek: oba około -0.96 — silna ujemna korelacja liniowa.')
print('Interpretacja biznesowa: każde 100 PLN podwyżki ceny spada sprzedaż o ~150 sztuk.')

### Mini-ćwiczenie B — branża marketing (uzupełnij __)

Ocena gwiazdkowa produktu (1–5) vs wartość sprzedaży. Sprawdź **monotoniczność** (Spearman) i **liniowość** (Pearson).

In [ ]:
# Faded — uzupełnij ___ 
np.random.seed(9)
oceny = np.random.choice([1, 2, 3, 4, 5], 200, p=[0.05, 0.1, 0.25, 0.4, 0.2])
# Sprzedaż rośnie z oceną, ale nie liniowo — produkt z oceną 5 sprzedaje 3x więcej niż z 4
sprzedaz = (oceny ** 1.8) * 100 + np.random.normal(0, 50, 200)

rp_oc, _ = pearsonr(oceny, sprzedaz)         # Pearson — UZUPEŁNIJ TO
rs_oc, _ = spearmanr(oceny, sprzedaz)        # Spearman — UZUPEŁNIJ TO

print(f"Ocena vs sprzedaż:  Pearson={rp_oc:+.3f}  Spearman={rs_oc:+.3f}")
print()
# Pytanie: dlaczego Spearman > Pearson? Co to mówi o relacji?

### Mini-ćwiczenie C — branża e-commerce (samodzielnie)

Czas spędzony na stronie produktu (sekundy) vs konwersja (czy klient kupił, 0/1). **Sam wybierz** korelację, sam wykonaj scatter, sam zinterpretuj.

In [ ]:
# Samodzielnie: czas na stronie vs konwersja
np.random.seed(13)
czas_s = np.random.exponential(60, 500).clip(5, 600)
# Konwersja: prawdopodobieństwo rośnie z czasem, ale saturuje
p_kup = 1 / (1 + np.exp(-(czas_s - 80) / 30))   # logistic
kupil = (np.random.random(500) < p_kup).astype(int)

# TODO 1: oblicz Pearson i Spearman dla (czas_s, kupil)
# TODO 2: który ma sens dla danych binarnych (0/1) vs ciągłych?
# TODO 3: w komentarzu zinterpretuj wynik biznesowo
# Wskazówka: konwersja to klasyczne 0/1 — ostrożnie z Pearsonem

---

## 5. scipy.stats — kompletny opis rozkładu

Do tej pory liczyliśmy poszczególne miary. **scipy.stats** ma jedną funkcję która daje wszystko: `describe()`.

**🔑 Kluczowe rozróżnienie: dystrybucja vs próba** (klatki vs film)

Dotąd patrzyliście na **liczby** (5, 7, 12, 9, ...). Dojrzały analityk patrzy na **kształt rozkładu**: czy są symetryczne, prawostronnie skośne, dwumodalne, ze spiczem? Liczby = pojedyncze klatki filmu. Dystrybucja = cały film. Możesz mieć identyczne 5 klatek z dwóch filmów (te same liczby), ale fabuły (rozkłady) zupełnie różne. To dokładnie problem Anscombe quartet (W09) i Datasaurusa (sekcja 0).

`scipy.stats.describe` daje **kształt** od razu: skośność (asymetria) + kurtozę (spiczastość/ogon).

**Brat z W05/W07:** w pandas znacie już `df.describe()` — pokazuje count, mean, std, min, kwartyle, max. `scipy.stats.describe()` to wersja **rozszerzona**: dodatkowo skośność i kurtoza (kształt rozkładu). Pandas opisuje co jest *w danych*; scipy opisuje *kształt rozkładu*.

Plus dwie miary kształtu rozkładu:

| Miara | Co mówi | Wartości typowe |
|-------|---------|-----------------|
| **Skośność** (skewness) | Asymetria rozkładu | 0 = symetryczny, >0 = ogon w prawo, <0 = w lewo |
| **Kurtoza** (kurtosis) | spiczastość/koncentracja rozkładu | 0 = normalny (Fisher), >0 = bardziej spiczasty + ciężkie ogony, <0 = spłaszczony |

In [ ]:
# scipy.stats.describe — wszystko naraz
from scipy.stats import describe, skew, kurtosis

opis = describe(df['wynagrodzenie'].dropna(), nan_policy='omit')
print("scipy.stats.describe():")
print(f"  N obserwacji: {opis.nobs}")
print(f"  min, max:     ({opis.minmax[0]:,}, {opis.minmax[1]:,})")
print(f"  Średnia:      {opis.mean:,.0f}")
print(f"  Wariancja:    {opis.variance:,.0f}")
print(f"  Skośność:     {opis.skewness:+.3f}")
print(f"  Kurtoza:      {opis.kurtosis:+.3f}")

**Dodatnia skośność** (typowa dla pensji, czasów odpowiedzi, cen domów): rozkład prawostronnie skośny — długi ogon wysokich wartości. **To jest naturalne**, nie błąd!

**🚨 Częsty błąd interpretacyjny:** „skośność = anomalia, do naprawienia”. Nie. Wynagrodzenia, ceny domów, czas oczekiwania — naturalnie skośne. Akceptuj rzeczywistość, raportuj medianę zamiast średniej.

**Kurtoza** w scipy zwraca `kurtosis - 3` (Fisher) — porównanie do rozkładu normalnego. Wartość 0 = jak normalny. Dodatnia = piki + ciężkie ogony (więcej skrajnych obserwacji niż „normalnie”). Ujemna = spłaszczony.

In [ ]:
# Cztery rozkłady — porównanie skośności i kurtozy
np.random.seed(0)
rozklady = {
    'Normalny': np.random.normal(0, 1, 1000),
    'Prawostronnie skośny': np.random.exponential(1, 1000),
    'Lewostronnie skośny': -np.random.exponential(1, 1000),
    'Spiczasty (t Studenta)': np.random.standard_t(df=3, size=1000),
}

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, (nazwa, dane) in zip(axes.flat, rozklady.items()):
    ax.hist(dane, bins=50, color='#5B9BD5', alpha=0.7, edgecolor='white')
    sk = skew(dane)
    ku = kurtosis(dane)
    ax.set_title(f"{nazwa}\nSkośność={sk:+.2f}  Kurtoza={ku:+.2f}")
    ax.grid(alpha=0.3)
fig.suptitle('Cztery rozkłady — co mówi skośność i kurtoza', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 6. Wartości odstające (outliery) — IQR i z-score

**Outlier** — pojęcie znacie już z W09 (Anscombe quartet) i W10 (boxplot pokazuje outliery jako kropki). Dziś **matematyzujemy** je — dwie konkretne metody wykrywania: IQR i z-score.

Definicja przypomnienie: obserwacja **nietypowa** w stosunku do reszty. Nasi 3 prezesi (CEO/CTO/CFO) z pensjami 25–32k przy medianie 7.5k.

**🚨 Częsty błąd interpretacyjny — szczególnie groźny dla biznesu:** „outlier to błąd, trzeba usunąć”.

**FAŁSZ.** Outlier to **sygnał** który wymaga decyzji:

| Sytuacja | Decyzja |
|----------|---------|
| Błąd wpisu (negatywna wiek = -25) | **Usuń** lub popraw |
| Pomiar awaryjny (czujnik się zepsuł) | **Usuń** + log |
| Top klient (1% przychodu) | **Zostaw** + analizuj osobno |
| CEO firmy (25k) | **Zostaw** — to fakt biznesowy |
| Oszustwo (*fraud* — np. wyłudzenie, podejrzana transakcja) | **Zostaw** + zgłoś |

**Nigdy „usuń bez patrzenia”.** To może wyrzucić najważniejsze obserwacje (np. wykrywanie oszustw — *fraud detection*!).

### Dwie metody wykrywania

**1. Metoda IQR (Tukey 1977)** — najpopularniejsza:
- Granica dolna: `Q1 - 1.5 · IQR`
- Granica górna: `Q3 + 1.5 · IQR`
- Wartości poza = outliery
- Odporna na same outliery (bo używa kwartyli, nie średniej)

**2. Metoda z-score (standaryzacja)** — dla rozkładów normalnych:
- z = (x − średnia) / std
- Outlier jeśli |z| > 3 (czyli ponad 3 odchylenia od średniej)
- **Uwaga:** sama używa średniej i std — które są wrażliwe na outliery!

In [ ]:
# Metoda 1: IQR
Q1 = placa.quantile(0.25)
Q3 = placa.quantile(0.75)
IQR = Q3 - Q1

granica_dolna = Q1 - 1.5 * IQR
granica_gorna = Q3 + 1.5 * IQR

outliery_iqr = df[(placa < granica_dolna) | (placa > granica_gorna)]

print("METODA IQR:")
print(f"  Q1 = {Q1:,.0f}, Q3 = {Q3:,.0f}, IQR = {IQR:,.0f}")
print(f"  Granica dolna: {granica_dolna:,.0f}")
print(f"  Granica górna: {granica_gorna:,.0f}")
print(f"  Wykryto outlierów: {len(outliery_iqr)}")
print()
print(outliery_iqr[['dzial', 'staz_lat', 'wynagrodzenie']])

In [ ]:
# Metoda 2: z-score (standaryzacja — odległość w jednostkach odchylenia standardowego)
# stats.zscore zwraca ndarray; opakowujemy w Series zeby zachowac indeks dla df
z_scores = pd.Series(np.abs(stats.zscore(placa)), index=df.index)
outliery_z = df[z_scores > 3].copy()
outliery_z['z_score'] = z_scores[z_scores > 3].round(2)

print("METODA Z-SCORE:")
print(f"  Próg: |z| > 3 (ponad 3 odchylenia standardowe od średniej)")
print(f"  Wykryto outlierów: {len(outliery_z)}")
print()
print(outliery_z[['dzial', 'staz_lat', 'wynagrodzenie', 'z_score']])

**Porównanie:** IQR zwykle wykrywa więcej outlierów niż z-score (bo z-score wymaga >3σ — bardzo dużej odległości). IQR jest bardziej czuły, z-score bardziej konserwatywny.

**Decyzja biznesowa**: nasze 3 outliery to CEO/CTO/CFO — **zostawiamy** (to fakt). Ale do raportu dla zarządu o „typowych pensjach” — analizujemy je **osobno** (kierownictwo) i robimy raport o medianie reszty (pracowników).

In [ ]:
# Wpływ outlierów na statystyki — ilustracja
df_bez_outlierow = df[~df.index.isin(outliery_iqr.index)]

print("=" * 60)
print(f"{'Statystyka':<20s} {'Z outlierami':>15s} {'Bez':>15s}")
print("=" * 60)
print(f"{'N obserwacji':<20s} {len(df):>15} {len(df_bez_outlierow):>15}")
print(f"{'Średnia (PLN)':<20s} {df['wynagrodzenie'].mean():>15,.0f} {df_bez_outlierow['wynagrodzenie'].mean():>15,.0f}")
print(f"{'Mediana (PLN)':<20s} {df['wynagrodzenie'].median():>15,.0f} {df_bez_outlierow['wynagrodzenie'].median():>15,.0f}")
print(f"{'Std (PLN)':<20s} {df['wynagrodzenie'].std():>15,.0f} {df_bez_outlierow['wynagrodzenie'].std():>15,.0f}")
print("=" * 60)

**Co widać** (porównaj liczby w tabeli powyżej):
- **Średnia** spada wyraźnie po usunięciu 3 outlierów — średnia jest **wrażliwa** na outliery
- **Mediana** prawie się nie zmieniła — **odporna** na outliery
- **Std** spada drastycznie — bo std liczy kwadraty odchyleń, outlier 25–32k podnosi to bardzo (proporcjonalnie więcej niż średnia)

To jest dokładnie powód dla którego **dla danych skośnych raportuje się medianę + IQR**, nie średnią + std.

### Mini-ćwiczenie A — branża logistyka (przykład gotowy)

Czas realizacji zamówienia (godziny) — 500 paczek. Część zamówień ma anomalia (awarie, święta). **Wykryjmy je** dwiema metodami i porównajmy.

In [ ]:
# Worked: outliery w czasach realizacji
np.random.seed(2)
czasy = np.concatenate([
    np.random.normal(48, 8, 480),       # 480 normalnych: ~48h ± 8h
    np.random.uniform(120, 200, 20),    # 20 awarii: 5–8 dni
])

Q1, Q3 = np.percentile(czasy, [25, 75])
iqr = Q3 - Q1
out_iqr = ((czasy < Q1 - 1.5*iqr) | (czasy > Q3 + 1.5*iqr)).sum()
out_z = (np.abs((czasy - czasy.mean()) / czasy.std()) > 3).sum()

print(f"Wykryte outliery (IQR):    {out_iqr}")
print(f"Wykryte outliery (z>3):    {out_z}")
print()
print('Wniosek: IQR czulszy (znalazł 18-22 awarii), z-score konserwatywny (10-13)')
print('Decyzja: te 20 awarii to NIE błąd danych — to fakt operacyjny. Zostaw + analizuj osobno.')

### Mini-ćwiczenie B — branża handel (uzupełnij __)

Wartości faktur dla klienta korporacyjnego. Większość 1k–10k PLN, kilka kontraktów strategicznych 50k+. **Wykryj outliery i zdecyduj** czy to anomalia czy „top customers” (kluczowi klienci).

In [ ]:
# Faded — uzupełnij ___
np.random.seed(4)
faktury = np.concatenate([
    np.random.exponential(2500, 380).clip(500, 12000),  # zwykłe
    np.random.uniform(45000, 90000, 20),                # kontrakty strategiczne
])

Q1 = np.percentile(faktury, 25)         # 25 percentyl — UZUPEŁNIJ TO
Q3 = np.percentile(faktury, 75)         # 75 percentyl — UZUPEŁNIJ TO
iqr = Q3 - Q1                            # rozstęp ćwiartkowy
granica_gorna = Q3 + 1.5 * iqr           # górna granica IQR

outliery_idx = np.where(faktury > granica_gorna)[0]
print(f"Wykryto {len(outliery_idx)} faktur powyżej {granica_gorna:,.0f} PLN")
print()
# Pytanie: czy te outliery to BŁĄD czy WAŻNI klienci?
# Odpowiedź wpływa na decyzję: usunąć / zostawić / analizować osobno

### Mini-ćwiczenie C — branża HR (samodzielnie)

Liczba dni nieobecności pracownika rocznie. Większość 0–15 dni, ale są wyjątki (długie zwolnienia, urlopy macierzyńskie). **Sam wybierz metodę i zdecyduj**.

In [ ]:
# Samodzielnie: nieobecności w ciągu roku
np.random.seed(15)
dni = np.concatenate([
    np.random.poisson(7, 195),               # zwykłe — kilka dni urlopu
    np.random.uniform(60, 180, 5),            # macierzyńskie / poważne choroby
]).astype(int)

# TODO 1: wykryj outliery metodą IQR (oblicz Q1, Q3, granicę górną)
# TODO 2: wykryj outliery metodą z-score (>3)
# TODO 3: zdecyduj — które obserwacje USUNĄĆ z analizy efektywności, a które ZOSTAWIĆ?
# Wskazówka: kontekst biznesowy decyduje — czy oceniamy pracowitość czy obecność?

---

## 7. Kulminacja — kompletna analiza HR dla zarządu

Wyobraź sobie: zarząd prosi o **raport statystyczny** o wynagrodzeniach. Termin: 1 tydzień. Co odpowiadasz?

Łączymy wszystkie 5 obszarów z dzisiejszego wykładu w **jeden raport**:

In [ ]:
# RAPORT STATYSTYCZNY HR — kompletna analiza
from scipy.stats import describe, skew, kurtosis, pearsonr, spearmanr

print("=" * 60)
print("   RAPORT STATYSTYCZNY HR — wynagrodzenia 2024")
print("=" * 60)

# 1. Tendencja centralna
print("\n1. TENDENCJA CENTRALNA")
print(f"   Średnia:    {placa.mean():,.0f} PLN")
print(f"   Mediana:    {placa.median():,.0f} PLN  ← raport oficjalny")
print(f"   Różnica:    {placa.mean() - placa.median():+,.0f} PLN (skośność prawostronna)")

# 2. Rozproszenie
print("\n2. ROZPROSZENIE")
print(f"   IQR (P25-P75): {placa.quantile(0.25):,.0f}–{placa.quantile(0.75):,.0f} PLN")
print(f"   P5 (najniżej): {placa.quantile(0.05):,.0f} PLN")
print(f"   P95 (top 5%):  {placa.quantile(0.95):,.0f} PLN")

# 3. Outliery
iqr_v = placa.quantile(0.75) - placa.quantile(0.25)
outliers_count = ((placa < placa.quantile(0.25) - 1.5*iqr_v) | (placa > placa.quantile(0.75) + 1.5*iqr_v)).sum()
print(f"\n3. OUTLIERY")
print(f"   Wykryto: {outliers_count} (kierownictwo — analiza osobna)")

# 4. Korelacje
rp, _ = pearsonr(df['staz_lat'], df['wynagrodzenie'])
rs, _ = spearmanr(df['staz_lat'], df['wynagrodzenie'])
print(f"\n4. KORELACJA STAŻ ↔ PENSJA")
print(f"   Pearson  r = {rp:+.2f}  (związek liniowy)")
print(f"   Spearman r = {rs:+.2f}  (związek monotoniczny)")

# 5. Kształt
print(f"\n5. KSZTAŁT ROZKŁADU")
print(f"   Skośność: {skew(placa):+.2f}  (>0: prawostronna — typowe dla pensji)")
print(f"   Kurtoza:  {kurtosis(placa):+.2f}  (>0: ciężkie ogony)")

p95 = placa.quantile(0.95)
kierownictwo = placa[placa > 20000].count()
print("\n" + "=" * 60)
print("WNIOSEK BIZNESOWY:")
print(f"  Mediana {placa.median():,.0f} PLN — typowy pracownik. Zarobki rosną ze stażem")
print(f"  (Pearson r={rp:+.2f}), ale z saturacją — po 10 latach pensja")
print(f"  nie rośnie liniowo (Spearman r={rs:+.2f} > Pearson).")
print(f"  Top 5% (>{p95:,.0f} PLN) i kierownictwo ({kierownictwo} osób >20 000 PLN)")
print(f"  wymagają osobnej analizy — uwzględnij w raportach segmentowych.")
print("=" * 60)

---

## 8. Pułapki Pythona / scipy / numpy

Siedem rzeczy, na które trzeba uważać. Każda z nich raz w karierze popełniona — kosztowała komuś raport / decyzję / pieniądze.

### Pułapka 1: `np.std()` vs `pandas.Series.std()` — różne ddof domyślne
Już omawiane w sekcji 1. Dla biznesu: pandas (n–1) jest właściwy.

### Pułapka 2: `mean()` z NaN
NumPy: `np.mean([1, 2, np.nan])` zwraca `nan`. Pandas: `.mean()` automatycznie pomija NaN. Sprawdź zawsze: `pd.isna().sum()` przed liczeniem.

### Pułapka 3: `scipy.stats.mode` zachowanie historyczne
W scipy <1.9: `mode` zwracał najmniejszą wartość przy remisie. W ≥1.9: najmniejszą jeśli `keepdims=True`. **Zawsze podawaj `keepdims` explicite** lub używaj `pd.Series.mode()`.

### Pułapka 4: `pearsonr()` zwraca obiekt (scipy ≥1.9), nie zwykły tuple
Idiomatycznie: `wynik = pearsonr(x, y); r = wynik.statistic; p = wynik.pvalue`. Stary kod `r, p = pearsonr(x, y)` nadal działa (kompatybilność wsteczna).

### Pułapka 5: kurtoza Fisher vs Pearson
scipy zwraca **kurtozę Fisher** (kurtoza – 3, normalny = 0). Wikipedia często podaje **Pearson** (normalny = 3). Sprawdź definicję.

### Pułapka 6: `pearsonr` z NaN — `ValueError`
scipy nie radzi sobie z NaN w danych — wyrzuci wyjątek. Pandas `df.corr()` automatycznie odrzuca pary z NaN (`min_periods` parametr). **Przed `pearsonr` zawsze:** `df.dropna(subset=['x','y'])`.

### Pułapka 7: `scipy.stats.zscore` zwraca ndarray (a nie Series)
Jeśli pracujesz z `df`, owijaj wynik: `pd.Series(stats.zscore(df['kol']), index=df.index)`. Inaczej tracisz indeks i alignement do pozostałych kolumn DataFrame.

In [ ]:
# Demonstracja pułapek — krótko
import warnings
warnings.filterwarnings('ignore')

# Pułapka 2: NaN
with_nan = np.array([1.0, 2.0, np.nan, 4.0])
print(f"np.mean z NaN:  {np.mean(with_nan)}      ← NaN!")
print(f"np.nanmean:     {np.nanmean(with_nan)}        ← OK")
print(f"pd.Series.mean: {pd.Series(with_nan).mean()} ← OK (auto skip NaN)")

# Pulapka 4: pearsonr zwraca obiekt (scipy >=1.9), nie zwykly tuple
wynik = pearsonr([1, 2, 3], [2, 4, 6])
print(f"\npearsonr zwraca: {wynik}")
print(f"  .statistic (r):  {wynik.statistic}")
print(f"  .pvalue:         {wynik.pvalue}")
# Lub idiomatycznie: r, p = pearsonr(x, y)  -- nadal dziala (kompatybilnosc)

---

## 9. Ściąga API — co zapamiętać

### Tendencja centralna

| Chcę... | Kod |
|---------|-----|
| Średnia | `df['kol'].mean()` |
| Mediana | `df['kol'].median()` |
| Dominanta | `df['kol'].mode().iloc[0]` |

### Rozproszenie

| Chcę... | Kod | Uwagi |
|---------|-----|-------|
| Wariancja | `df['kol'].var()` | ddof=1 (próba) |
| Std | `df['kol'].std()` | ddof=1 |
| Rozstęp | `df['kol'].max() - df['kol'].min()` | |
| Percentyle | `df['kol'].quantile([0.25, 0.5, 0.75])` | |
| IQR | `df['kol'].quantile(0.75) - df['kol'].quantile(0.25)` | |

### Korelacja

| Chcę... | Kod |
|---------|-----|
| Pearson | `from scipy.stats import pearsonr; r, p = pearsonr(x, y)` |
| Spearman | `from scipy.stats import spearmanr; r, p = spearmanr(x, y)` |
| Macierz korelacji | `df.corr()` (Pearson) lub `df.corr(method='spearman')` |

### Kształt rozkładu

| Chcę... | Kod |
|---------|-----|
| Wszystko naraz | `from scipy.stats import describe; describe(arr)` |
| Skośność | `from scipy.stats import skew; skew(arr)` |
| Kurtoza | `from scipy.stats import kurtosis; kurtosis(arr)` |

### Outliery

| Chcę... | Kod |
|---------|-----|
| z-score | `from scipy.stats import zscore; np.abs(zscore(arr)) > 3` |
| IQR test | `Q1, Q3 = arr.quantile([0.25, 0.75]); ((arr < Q1-1.5*(Q3-Q1)) \| (arr > Q3+1.5*(Q3-Q1)))` |

### Quick-and-dirty

| Chcę... | Kod |
|---------|-----|
| Wszystkie podstawowe | `df.describe()` |
| Korelacja całej tabeli | `df.corr()` |
| Statystyki per grupa | `df.groupby('kategoria').describe()` |

---

## 10. Podgląd laboratorium

Na laboratorium przećwiczycie wszystkie 5 obszarów na **datasecie kadrowym** (HR — 200 pracowników, 5 działów). Inny dataset niż dziś (z dodanymi outlierami i ocenami), żeby przetestować wszystkie narzędzia w kontrolowanym środowisku.

**Start — 3 komendy:**
```
cd C:\Users\student\python2
.venv\Scripts\Activate.ps1
code .
```

**Ćwiczenia na labie (≈85 min):**

- **Ćwiczenie 1** *(20 min)* — statystyki opisowe (centralne + rozproszenie + per dział + histogram)
- **Ćwiczenie 2** *(20 min)* — korelacje Pearson vs Spearman; macierz korelacji + scatter z trendem
- **Ćwiczenie 3** *(30 min)* — pełna analiza statystyczna (`scipy.stats.describe`, percentyle, per dział, boxplot)
- **Ćwiczenie 4** *(15 min)* — outliery (IQR, z-score, wpływ na statystyki) + commit

Utworzysz notebook `lab11_statystyka.ipynb`. Commity po każdym ćwiczeniu. Zero niespodzianek — wszystkie komendy znasz z dzisiejszego wykładu.

---

## Na koniec

> *„Far better an approximate answer to the right question, which is often vague, than an exact answer to the wrong question, which can always be made precise.”*
> *(„O wiele lepiej mieć przybliżoną odpowiedź na właściwe pytanie — często niejasne — niż dokładną odpowiedź na pytanie złe, które zawsze da się sformułować precyzyjnie.”)*
> — John W. Tukey, *The Future of Data Analysis* (1962)

John Tukey to ojciec **statystyki eksploracyjnej** (Exploratory Data Analysis — EDA — eksploracyjna analiza danych). Wymyślił **boxplot** (poznany w W10), zdefiniował **outlier 1.5×IQR** (dziś), spopularyzował **scatter plot** jako pierwsze narzędzie analityka.

Tukey ostrzega: średnia, korelacja, percentyl — same w sobie nic nie znaczą. Liczą się **dopiero w kontekście pytania biznesowego**, na które odpowiadają. Dlatego kolejność: pytanie → dane → liczba → decyzja. Niczego mniej.